# 3.8 — One-Hot Encoding

এই section-এ categorical feature-গুলোকে সংখ্যায় রূপান্তর করব, যাতে Machine Learning model সেগুলো ব্যবহার করতে পারে। এখানে `DictVectorizer` দিয়ে One-Hot Encoding করা হয়েছে।

> **ধারণা বুঝতে সমস্যা হলে:** Correlation-এর detailed theory তোমার আলাদা Notion note-এ আছে। এই notebook-এ শুধু project-এর প্রয়োজনীয় concept ও code রাখা হয়েছে।

**এই code-গুলো আগের section-এর `df_train`, `df_val`, `categorical` এবং `numerical` variables ব্যবহার করছে।**

## 1. কেন Encoding দরকার?

আমাদের numerical features—`tenure`, `monthlycharges`, `totalcharges`—আগেই সংখ্যা। কিন্তু `contract`-এ `month-to-month`, `one_year`, `two_year`-এর মতো text আছে। Model সরাসরি এই text-গুলোকে numerical calculation-এ ব্যবহার করতে পারে না, তাই encode করতে হয়।

### One-Hot Encoding কী করে?

একটি categorical column-এর প্রতিটি category-এর জন্য আলাদা `0/1` column তৈরি করে। যে category-টি row-তে আছে, তার column-এ `1`; বাকিগুলোতে `0` থাকে।

| `contract` | `contract=month-to-month` | `contract=one_year` | `contract=two_year` |
|---|---:|---:|---:|
| `two_year` | 0 | 0 | 1 |
| `month-to-month` | 1 | 0 | 0 |
| `one_year` | 0 | 1 | 0 |

**কেন `0, 1, 2` দিয়ে encode করি না?** কারণ তাতে model ভুল করে ভাবতে পারে `two_year` সংখ্যাগতভাবে `one_year`-এর চেয়ে বড় বা `month-to-month`-এর সঙ্গে কোনো numerical distance আছে। One-Hot Encoding এমন কৃত্রিম order তৈরি করে না।

Numerical features-গুলো আগের মতোই সংখ্যায় থাকবে; encoding মূলত categorical features-কে বদলায়।

## 2. Row-গুলোকে dictionary-তে রূপান্তর


In [ ]:
train_dict = df_train[categorical + numerical].to_dict(orient='records')
train_dict[:2]


`DictVectorizer` প্রতিটি row-কে dictionary হিসেবে নেয়। Pandas-এর `to_dict(orient='records')` এই format তৈরি করে—প্রতি row-এর জন্য একটি dictionary।


- `categorical + numerical` → সব input feature নির্বাচন করে; `churn` target অন্তর্ভুক্ত করে না।
- `orient='records'` → প্রতিটি row-কে একটি dictionary বানায়।

উদাহরণস্বরূপ, একটি row এমন হতে পারে:

```python
{
    'contract': 'two_year',
    'tenure': 72,
    'monthlycharges': 115.5
}
```

এখানে categorical value text, আর numerical value সংখ্যা হিসেবেই আছে।

## 3. `DictVectorizer` দিয়ে encode করা



প্রথমে transformer import ও তৈরি করি:



In [ ]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)


- `DictVectorizer` dictionary-র categorical values থেকে encoded columns তৈরি করে।
- `sparse=False` দিলে ফলাফল সাধারণ dense NumPy array হয়। Default sparse format memory বাঁচাতে পারে, কিন্তু dense array এখানে সহজে দেখা ও বোঝা যায়।



এবার training data-তে fit ও transform দুটোই করি:




In [ ]:
X_train = dv.fit_transform(train_dict)


**`fit_transform()` কী করে?**

1. `fit` → training data দেখে কোন কোন category আছে, এবং কোন output column তৈরি হবে তা শেখে।
2. `transform` → প্রতিটি row-কে সংখ্যার matrix-এ বদলায়।

`X_train` হলো model-এর input matrix। এখানে row-এর সংখ্যা customer-এর সংখ্যার সমান; column-এর সংখ্যা encoding-এর পর তৈরি হওয়া feature-এর সংখ্যার সমান।

In [ ]:
X_train.shape

এই dataset-এ output সাধারণত `(4225, 45)`:

- `4225` → training customer/row-এর সংখ্যা।
- `45` → encoding-এর পরে মোট input feature/column-এর সংখ্যা।

কারণ প্রতিটি categorical category আলাদা column পেতে পারে, তাই output column-এর সংখ্যা আগের column count-এর চেয়ে বেশি হয়।

## 4. তৈরি হওয়া column-এর নাম দেখা


In [ ]:
dv.get_feature_names_out()



এটি encoded matrix-এর column names দেখায়, যেমন `contract=two_year`, `internetservice=fiber_optic` ইত্যাদি। Numerical feature-এর নামও তালিকায় থাকবে।

এভাবে প্রতিটি output column কী বোঝায়, তা বোঝা যায়।

## 5. Validation data encode করা



Validation set-এর জন্য নতুন করে `fit` করব না। Training-এ শেখা একই category ও column mapping ব্যবহার করব।




In [ ]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

**কেন শুধু `transform()`?**

- Training ও validation matrix-এ একই columns এবং একই order থাকতে হবে।
- Validation-এ আবার `fit` করলে column mapping আলাদা হতে পারে।
- Model training-এর সময় যে feature structure দেখেছে, validation-এও সেই structure থাকা দরকার।

**নিয়ম:** Training data-তে `fit_transform()`; validation/test data-তে একই fitted transformer দিয়ে শুধু `transform()`।

## Quick Recap

| Code | কাজ |
|---|---|
| `.to_dict(orient='records')` | প্রতিটি row-কে dictionary বানায় |
| `DictVectorizer(sparse=False)` | dictionary data থেকে dense feature matrix তৈরি করার transformer |
| `dv.fit_transform(train_dict)` | training data-তে categories শেখে এবং encode করে |
| `dv.get_feature_names_out()` | output columns-এর নাম দেখায় |
| `dv.transform(val_dict)` | validation data-কে training-এর একই mapping দিয়ে encode করে |

**মূল কথা:** One-Hot Encoding categorical text-কে একাধিক `0/1` feature column-এ বদলায়। `DictVectorizer` একই সঙ্গে এই encoding করে এবং numerical features-গুলোকে unchanged অবস্থায় matrix-তে রাখে।

**Related:** Scikit-Learn-এর `OneHotEncoder` ও feature scaling নিয়ে আলাদা `notebook-scaling-ohe.ipynb` আছে।